# SHAP Feature Importance — XGBoost Explainability

Use **SHAP (SHapley Additive exPlanations)** to understand which features drive the XGBoost model's solar power predictions.

Requires `solar_xgb_model.json` produced by `02_xgboost_model.ipynb`.

**Plots generated:**
1. `shap_bar.png` — mean absolute SHAP value per feature (bar chart)
2. `shap_dot.png` — SHAP summary dot plot (impact direction + magnitude)
3. `shap_dependence_irradiance.png` — irradiance SHAP vs irradiance value, coloured by ambient_temp

In [ ]:
!pip install xgboost shap plotly influxdb-client --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import shap
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.dpi'] = 150
shap.initjs()
print('Libraries loaded.')

In [ ]:
# ── Load data and model ────────────────────────────────────────────────────────
%run data_loader.py
df = load_solar_data()

# Reconstruct the exact same feature matrix and 80/20 split as notebook 02
FEATURES = ['hour', 'month', 'day_of_year',
            'irradiance', 'ambient_temp', 'module_temp']
TARGET   = 'power_now_w'

df_model = df[FEATURES + [TARGET, 'datetime']].dropna().copy()
df_model['datetime'] = pd.to_datetime(df_model['datetime'])
df_model = df_model.sort_values('datetime').reset_index(drop=True)

X = df_model[FEATURES].values
y = df_model[TARGET].values

split_idx = int(len(df_model) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Wrap in DataFrames so SHAP preserves feature names
X_test_df  = pd.DataFrame(X_test,  columns=FEATURES)
X_train_df = pd.DataFrame(X_train, columns=FEATURES)

# Load saved model
model = XGBRegressor()
model.load_model('solar_xgb_model.json')

print('Model loaded.')
print(f'Test set size: {X_test_df.shape}')

In [ ]:
# ── Compute SHAP values ────────────────────────────────────────────────────────
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_df)

print(f'SHAP values shape: {shap_values.shape}')  # (n_test_samples, n_features)

In [ ]:
# ── Plot 1: SHAP Summary Bar — top 10 features ────────────────────────────────
shap.summary_plot(
    shap_values, X_test_df,
    plot_type='bar',
    max_display=10,
    show=False
)
plt.title('SHAP Feature Importance (Mean |SHAP value|)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shap_bar.png')

In [ ]:
# ── Plot 2: SHAP Summary Dot plot ─────────────────────────────────────────────
shap.summary_plot(
    shap_values, X_test_df,
    max_display=10,
    show=False
)
plt.title('SHAP Summary — Feature Impact on Predictions', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_dot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shap_dot.png')

In [ ]:
# ── Plot 3: SHAP Dependence — irradiance (interaction: ambient_temp) ──────────
shap.dependence_plot(
    'irradiance', shap_values, X_test_df,
    interaction_index='ambient_temp',
    show=False
)
plt.title(
    'SHAP Dependence: Irradiance  (interaction: ambient_temp)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('shap_dependence_irradiance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shap_dependence_irradiance.png')

## What SHAP Values Mean — Plain English for a Solar System Owner

**SHAP (SHapley Additive exPlanations)** values answer one question:  
*"How much did each input push today's power prediction higher or lower compared to the average?"*

Think of the average predicted output as a baseline (say, 800 W at solar noon).  
The SHAP values for any specific 15-minute reading then show the individual contribution of each feature:

| Feature | Example SHAP Value | Meaning |
|---|---|---|
| `irradiance = 950 W/m²` | +420 W | High sun intensity **boosted** the prediction by 420 W |
| `ambient_temp = 42 °C` | −85 W | Hot weather **reduced** the prediction — panels lose efficiency |
| `hour = 14` | +30 W | Afternoon sun angle gives a small positive effect |
| `module_temp = 65 °C` | −60 W | Hot panel surface further **reduces** output |

**Key takeaways for your 3.57 kW Karnal system:**

1. **Irradiance dominates** — the single biggest driver of output is how much sunlight reaches your panels. Dust, clouds, or shade will appear as large negative SHAP values.

2. **Temperature hurts** — high ambient and module temperatures reduce silicon PV efficiency (roughly −0.4 %/°C above 25 °C STC). In Karnal summers (May–Jun) you may see 5–10 % less output than on an equally sunny but cooler day.

3. **Time features capture seasonality** — `hour`, `month`, and `day_of_year` encode the sun's changing angle through the day and across seasons so the model can distinguish a short winter noon from a long summer afternoon.

4. **Reading the dot plot** — each dot is one 15-minute reading. Dot colour = feature value (red = high, blue = low). A cloud of red dots on the right side of the `irradiance` row confirms: high irradiance always drives predictions up.

5. **Reading the dependence plot** — the temperature-coloured scatter on the irradiance dependence chart reveals a fan shape at high irradiance values: hotter days (red dots) sit lower on the SHAP axis, quantifying exactly how much heat is stealing from your potential peak output.